# **Importación del dataset EMG**

In [20]:
# Importación de librerías
import numpy as np
import biosignalsnotebooks as bsnb
import pickle
from bokeh.plotting import figure, show


with open('dataset_EMG.pkl', 'rb') as f:
    emg = pickle.load(f)
Fs=100

len(np.concatenate(emg['base'][np.where(emg['target'] == 0)]))

1300

| Descripción | Categoría |
|----------|----------|
|Señal en reposo (mano)                     |0|
|Señal en flexión leve (mano)               |1|
|Señal en flexión brusca (mano)             |2|
|Señal en contracción isométrica (mano)     |3|
|Señal en reposo (tobillo)                  |4|
|Señal en movimiento leve (tobillo)         |5|

# **Tratamiento EMG (mano)**

In [21]:
target_values = [0, 1, 2, 3]
indices = np.isin(emg['target'], target_values)
emg_signal=np.concatenate(emg['base'][indices])
time = bsnb.generate_time(emg_signal,Fs)

bokeh_figure = figure(x_axis_label='Time (s)', y_axis_label='Raw Data',title='Señal en reposo (mano)')
bokeh_figure.line(time,emg_signal,legend_label='Datos Originales')
show(bokeh_figure)

In [22]:
# Scientific packages
from scipy.signal import butter, lfilter
from scipy.stats import linregress

# el dataset ya tiene la eliminación de la línea base

## Filtrado

In [23]:
low_cutoff = 10 # Hz
# high_cutoff = 300 # Hz no se considera debido a que la Fs del dataset es 100
pre_pro_signal = lfilter(*butter(6,low_cutoff,'highpass',fs=Fs),emg_signal)

## Aplicación del operador TKEO

In [24]:
tkeo = []
for i in range(0, len(pre_pro_signal)):
    if i == 0 or i == len(pre_pro_signal) - 1:
        tkeo.append(pre_pro_signal[i])
    else:
        tkeo.append(np.power(pre_pro_signal[i], 2) - (pre_pro_signal[i + 1] * pre_pro_signal[i - 1]))

# Smoothing level [Size of sliding window used during the moving average process (a function of sampling frequency)]
smoothing_level_perc = 20 # Percentage.
smoothing_level = int((smoothing_level_perc / 100) * Fs)

rect_signal = np.absolute(tkeo)

## Smoothing Phase

In [25]:
# Smoothing level [Size of sliding window used during the moving average process (a function of sampling frequency)]
smoothing_level_perc = 20 # Percentage.
smoothing_level = int((smoothing_level_perc / 100) * Fs)

# [Signal Rectification]
rect_signal = np.absolute(tkeo)

# [First Moving Average Filter]
rect_signal = bsnb.aux_functions._moving_average(rect_signal, Fs / 10)

# [Second Smoothing Phase]
smooth_signal = []
for i in range(0, len(rect_signal)):
    if smoothing_level < i < len(rect_signal) - smoothing_level:
        smooth_signal.append(np.mean(rect_signal[i - smoothing_level:i + smoothing_level]))
    else:
        smooth_signal.append(0)

## Detección del límite de detección

In [26]:
# [Threshold]
avg_pre_pro_signal = np.average(pre_pro_signal)
std_pre_pro_signal = np.std(pre_pro_signal)

# Regression function.
def normReg(thresholdLevel):
    threshold_0_perc_level = (- avg_pre_pro_signal) / float(std_pre_pro_signal)
    threshold_100_perc_level = (max(smooth_signal) - avg_pre_pro_signal) / float(std_pre_pro_signal)
    m, b = linregress([0, 100], [threshold_0_perc_level, threshold_100_perc_level])[:2]
    return m * thresholdLevel + b

# Chosen Threshold Level (Example with two extreme values)
threshold_level = 10 # % Relative to the average value of the smoothed signal
threshold_level_norm_10 = normReg(threshold_level)

threshold_level = 80 # % Relative to the average value of the smoothed signal
threshold_level_norm_80 = normReg(threshold_level)

threshold_10 = avg_pre_pro_signal + threshold_level_norm_10 * std_pre_pro_signal
threshold_80 = avg_pre_pro_signal + threshold_level_norm_80 * std_pre_pro_signal

## Binarización de la señal suavizada

In [27]:
# Generation of a square wave reflecting the activation and inactivation periods.
binary_signal = []
for i in range(0, len(time)):
    if smooth_signal[i] >= threshold_10:
        binary_signal.append(1)
    else:
        binary_signal.append(0)

## Inicio y fin de los periodos de activación

In [28]:
#función modificada

def moving_average(data, wind_size=3):
    """
    -----
    Brief
    -----
    Application of a moving average filter for signal smoothing.

    -----------
    Description
    -----------
    In certain situations it will be interesting to simplify a signal, particularly in cases where
    some events with a random nature take place (the random nature of EMG activation periods is
    a good example).

    One possible simplification procedure consists in smoothing the signal in order to obtain
    only an "envelope". With this methodology the analysis is mainly centered on seeing patterns
    in data and excluding noise or rapid events [1].

    The simplification can be achieved by segmenting the time series in multiple windows and
    from each window an average value of all the samples that it contains will be determined
    (dividing the sum of all sample values by the window size).

    A quick and efficient implementation (chosen in biosignalsnotebooks package) of the moving window
    methodology is through a cumulative sum array.

    [1] https://en.wikipedia.org/wiki/Smoothing

    ---------
    Parameters
    ----------
    data : list
        List of signal samples.
    wind_size : int
        Number of samples inside the moving average window (a bigger value implies a smoother
        output signal).

    Returns
    -------
    out : numpy array
        Array that contains the samples of the smoothed signal.
    """

    wind_size = int(wind_size)
    ret = np.cumsum(data, dtype=float)
    ret[wind_size:] = ret[wind_size:] - ret[:-wind_size]
    return np.concatenate((np.zeros(wind_size - 1), ret[wind_size - 1:] / wind_size))

def thres_norm_reg(threshold_level, signal, pre_smooth_signal):
    """
    Regression function that with a percent input gives an absolute value of the threshold
    level (used in the muscular activation detection algorithm).
    Converts a relative threshold level to an absolute value.

    ----------
    Parameters
    ----------
    threshold_level : int
        Percentage value that defines the absolute threshold level relatively to the maximum value
        of signal.
    signal : list
        List of EMG smoothed signal samples.
    pre_smooth_signal : list
        Original EMG samples.

    Returns
    -------
    out : float
        Threshold level in absolute format.

     """
    avg_signal = np.average(pre_smooth_signal)
    std_signal = np.std(pre_smooth_signal)

    threshold_0_perc_level = (-avg_signal) / float(std_signal)
    threshold_100_perc_level = (np.max(signal) - avg_signal) / float(std_signal)

    slope, b_coeff = linregress([0, 100], [threshold_0_perc_level, threshold_100_perc_level])[:2]
    return slope * threshold_level + b_coeff

def modified_detect_emg_activations(emg_signal, sample_rate, smooth_level=20, threshold_level=10,
                           time_units=False, volts=False, resolution=None, device="biosignalsplux",
                           plot_result=False):
    """
    -----
    Brief
    -----
    Python implementation of Burst detection algorithm using Teager Kaiser Energy Operator.

    -----------
    Description
    -----------
    Activation events in EMG readings correspond to an increase of muscular activity, namely, from inaction to action.
    These events are characterised by an increase in electric potential that returns to the initial values when the
    muscle returns to a state of inaction.

    This function detects activation events using the Teager Kaiser Energy Operator.

    ----------
    Parameters
    ----------
    emg_signal : list
        List of EMG acquired samples.

    sample_rate : int
        Sampling frequency.

    smooth_level : number
        Defines a percentage proportional to the smoothing level, i.e. the bigger this value is,
        the more smoothed is the signal.

    threshold_level : number
        Specification of the single threshold position, used for distinguishing between activation
        (above) and inactivation samples (below).

    time_units : boolean
        If True this function will return the Burst begin and end positions in seconds.

    volts : boolean
        If True, then the conversion of raw units to mV will be done. Resolution need to be
        specified.

    resolution : int
        Selected resolution for data acquisition.

    device : str
        Specification of the device category.

    plot_result : boolean
        If True it will be presented a graphical representation of the detected burst in the EMG
        signal.

    Returns
    -------
    out : bursts begin (ndarray), bursts end (ndarray)
        Begin and end of bursts (sample number or time instant in seconds).

    smooth_signal: list
        It is returned the smoothed EMG signal (after the processing steps intended to simplify the
        signal).

    threshold_level: float
        The value of the detection threshold used to locate the begin and end of each muscular
        activation period.
    """

    if volts is True:
        if resolution is not None:
            emg_signal = bsnb.raw_to_phy("EMG", device, emg_signal, resolution, option="mV")
            units = "mV"
        else:
            raise RuntimeError(
                "For converting raw units to mV is mandatory the specification of acquisition "
                "resolution.")
    else:
        units = "Input Units"

    if time_units is True:
        time_units_str = "Time (s)"
        time = np.linspace(0, len(emg_signal) / sample_rate, len(emg_signal))
    else:
        time = np.linspace(0, len(emg_signal) - 1, len(emg_signal))
        time_units_str = "Sample Number"

    # ----------------------------------- Baseline Removal -----------------------------------------
    pre_pro_signal = np.array(emg_signal) - np.average(emg_signal)

    # ------------------------------------ Signal Filtering ----------------------------------------
    low_cutoff = 10  # Hz
    #high_cutoff = 300  # Hz

    # Application of the signal to the filter.
    pre_pro_signal = lfilter(*butter(6,low_cutoff,'highpass',fs=Fs),pre_pro_signal)

    # ------------------------------ Application of TKEO Operator ----------------------------------
    tkeo = []
    for i, signal_sample in enumerate(pre_pro_signal):
        if i in (0, len(pre_pro_signal) - 1):
            tkeo.append(signal_sample)
        else:
            tkeo.append(np.power(signal_sample, 2) - (pre_pro_signal[i + 1] *
                                                         pre_pro_signal[i - 1]))

    # Smoothing level - Size of sliding window used during the moving average process (a function
    # of sampling frequency)
    smoothing_level = int((smooth_level / 100) * sample_rate)

    # --------------------------------- Signal Rectification ---------------------------------------
    rect_signal = np.absolute(tkeo)

    # ------------------------------ First Moving Average Filter -----------------------------------
    rect_signal = moving_average(rect_signal, sample_rate / 10)

    # -------------------------------- Second Smoothing Phase --------------------------------------
    smooth_signal = []
    for i in range(0, len(rect_signal)):
        if smoothing_level < i < len(rect_signal) - smoothing_level:
            smooth_signal.append(np.mean(rect_signal[i - smoothing_level:i + smoothing_level]))
        else:
            smooth_signal.append(0)

    # ----------------------------------- Threshold -----------------------------------------------
    avg_pre_pro_signal = np.average(pre_pro_signal)
    std_pre_pro_signal = np.std(pre_pro_signal)

    threshold_level = avg_pre_pro_signal + thres_norm_reg(threshold_level, smooth_signal,
                                                           pre_pro_signal) * std_pre_pro_signal

    # Generation of a square wave reflecting the activation and inactivation periods.
    binary_signal = []
    for i in range(0, len(time)):
        if smooth_signal[i] >= threshold_level:
            binary_signal.append(1)
        else:
            binary_signal.append(0)

    # ------------------------------ Begin and End of Bursts --------------------------------------
    diff_signal = np.diff(binary_signal)
    act_begin = np.where(diff_signal == 1)[0]
    act_end = np.where(diff_signal == -1)[0]

    if time_units is True:
        time_begin = np.array(time)[act_begin]
        time_end = np.array(time)[act_end]
    else:
        time_begin = act_begin
        time_end = act_end

    # If plot is invoked by plot_result flag, then a graphical representation of the R peaks is
    # presented to the user.
    if plot_result is True:
        bsnb.plot([list(time), list(time)], [list(emg_signal), list(np.array(binary_signal) *
                                                               np.max(emg_signal))],
             yAxisLabel=["Data Samples (" + units + ")"] * 2,
             x_axis_label=time_units_str, legend_label=["EMG Signal", "Activation Signal"])

    return time_begin, time_end, smooth_signal, threshold_level

In [29]:
diff_signal = np.diff(binary_signal)
act_begin = np.where(diff_signal == 1)[0]
act_end = np.where(diff_signal == -1)[0]


activation_data = modified_detect_emg_activations(emg_signal, Fs, smooth_level=20, threshold_level=10, time_units=True, volts=False, resolution=None, device="bitalino_rev", plot_result=True)

# **Tratamiento EMG (tobillo)**

In [30]:
target_values = [4, 5]
indices = np.isin(emg['target'], target_values)
emg_signal=np.concatenate(emg['base'][indices])
time = bsnb.generate_time(emg_signal,Fs)

bokeh_figure = figure(x_axis_label='Time (s)', y_axis_label='Raw Data',title='Señal en reposo (mano)')
bokeh_figure.line(time,emg_signal,legend_label='Datos Originales')
show(bokeh_figure)

In [31]:
# Scientific packages
from scipy.signal import butter, lfilter
from scipy.stats import linregress

# el dataset ya tiene la eliminación de la línea base

## Filtrado

In [32]:
low_cutoff = 10 # Hz
# high_cutoff = 300 # Hz no se considera debido a que la Fs del dataset es 100
pre_pro_signal = lfilter(*butter(6,low_cutoff,'highpass',fs=Fs),emg_signal)

## Aplicación del operador TKEO

In [33]:
tkeo = []
for i in range(0, len(pre_pro_signal)):
    if i == 0 or i == len(pre_pro_signal) - 1:
        tkeo.append(pre_pro_signal[i])
    else:
        tkeo.append(np.power(pre_pro_signal[i], 2) - (pre_pro_signal[i + 1] * pre_pro_signal[i - 1]))

# Smoothing level [Size of sliding window used during the moving average process (a function of sampling frequency)]
smoothing_level_perc = 20 # Percentage.
smoothing_level = int((smoothing_level_perc / 100) * Fs)

rect_signal = np.absolute(tkeo)

## Smoothing Phase

In [34]:
# Smoothing level [Size of sliding window used during the moving average process (a function of sampling frequency)]
smoothing_level_perc = 20 # Percentage.
smoothing_level = int((smoothing_level_perc / 100) * Fs)

# [Signal Rectification]
rect_signal = np.absolute(tkeo)

# [First Moving Average Filter]
rect_signal = bsnb.aux_functions._moving_average(rect_signal, Fs / 10)

# [Second Smoothing Phase]
smooth_signal = []
for i in range(0, len(rect_signal)):
    if smoothing_level < i < len(rect_signal) - smoothing_level:
        smooth_signal.append(np.mean(rect_signal[i - smoothing_level:i + smoothing_level]))
    else:
        smooth_signal.append(0)

## Detección del límite de detección

In [35]:
# [Threshold]
avg_pre_pro_signal = np.average(pre_pro_signal)
std_pre_pro_signal = np.std(pre_pro_signal)

# Regression function.
def normReg(thresholdLevel):
    threshold_0_perc_level = (- avg_pre_pro_signal) / float(std_pre_pro_signal)
    threshold_100_perc_level = (max(smooth_signal) - avg_pre_pro_signal) / float(std_pre_pro_signal)
    m, b = linregress([0, 100], [threshold_0_perc_level, threshold_100_perc_level])[:2]
    return m * thresholdLevel + b

# Chosen Threshold Level (Example with two extreme values)
threshold_level = 10 # % Relative to the average value of the smoothed signal
threshold_level_norm_10 = normReg(threshold_level)

threshold_level = 80 # % Relative to the average value of the smoothed signal
threshold_level_norm_80 = normReg(threshold_level)

threshold_10 = avg_pre_pro_signal + threshold_level_norm_10 * std_pre_pro_signal
threshold_80 = avg_pre_pro_signal + threshold_level_norm_80 * std_pre_pro_signal

## Binarización de la señal suavizada

In [36]:
# Generation of a square wave reflecting the activation and inactivation periods.
binary_signal = []
for i in range(0, len(time)):
    if smooth_signal[i] >= threshold_10:
        binary_signal.append(1)
    else:
        binary_signal.append(0)

## Inicio y fin de los periodos de activación

In [37]:
#función modificada

def moving_average(data, wind_size=3):
    """
    -----
    Brief
    -----
    Application of a moving average filter for signal smoothing.

    -----------
    Description
    -----------
    In certain situations it will be interesting to simplify a signal, particularly in cases where
    some events with a random nature take place (the random nature of EMG activation periods is
    a good example).

    One possible simplification procedure consists in smoothing the signal in order to obtain
    only an "envelope". With this methodology the analysis is mainly centered on seeing patterns
    in data and excluding noise or rapid events [1].

    The simplification can be achieved by segmenting the time series in multiple windows and
    from each window an average value of all the samples that it contains will be determined
    (dividing the sum of all sample values by the window size).

    A quick and efficient implementation (chosen in biosignalsnotebooks package) of the moving window
    methodology is through a cumulative sum array.

    [1] https://en.wikipedia.org/wiki/Smoothing

    ---------
    Parameters
    ----------
    data : list
        List of signal samples.
    wind_size : int
        Number of samples inside the moving average window (a bigger value implies a smoother
        output signal).

    Returns
    -------
    out : numpy array
        Array that contains the samples of the smoothed signal.
    """

    wind_size = int(wind_size)
    ret = np.cumsum(data, dtype=float)
    ret[wind_size:] = ret[wind_size:] - ret[:-wind_size]
    return np.concatenate((np.zeros(wind_size - 1), ret[wind_size - 1:] / wind_size))

def thres_norm_reg(threshold_level, signal, pre_smooth_signal):
    """
    Regression function that with a percent input gives an absolute value of the threshold
    level (used in the muscular activation detection algorithm).
    Converts a relative threshold level to an absolute value.

    ----------
    Parameters
    ----------
    threshold_level : int
        Percentage value that defines the absolute threshold level relatively to the maximum value
        of signal.
    signal : list
        List of EMG smoothed signal samples.
    pre_smooth_signal : list
        Original EMG samples.

    Returns
    -------
    out : float
        Threshold level in absolute format.

     """
    avg_signal = np.average(pre_smooth_signal)
    std_signal = np.std(pre_smooth_signal)

    threshold_0_perc_level = (-avg_signal) / float(std_signal)
    threshold_100_perc_level = (np.max(signal) - avg_signal) / float(std_signal)

    slope, b_coeff = linregress([0, 100], [threshold_0_perc_level, threshold_100_perc_level])[:2]
    return slope * threshold_level + b_coeff

def modified_detect_emg_activations(emg_signal, sample_rate, smooth_level=20, threshold_level=10,
                           time_units=False, volts=False, resolution=None, device="biosignalsplux",
                           plot_result=False):
    """
    -----
    Brief
    -----
    Python implementation of Burst detection algorithm using Teager Kaiser Energy Operator.

    -----------
    Description
    -----------
    Activation events in EMG readings correspond to an increase of muscular activity, namely, from inaction to action.
    These events are characterised by an increase in electric potential that returns to the initial values when the
    muscle returns to a state of inaction.

    This function detects activation events using the Teager Kaiser Energy Operator.

    ----------
    Parameters
    ----------
    emg_signal : list
        List of EMG acquired samples.

    sample_rate : int
        Sampling frequency.

    smooth_level : number
        Defines a percentage proportional to the smoothing level, i.e. the bigger this value is,
        the more smoothed is the signal.

    threshold_level : number
        Specification of the single threshold position, used for distinguishing between activation
        (above) and inactivation samples (below).

    time_units : boolean
        If True this function will return the Burst begin and end positions in seconds.

    volts : boolean
        If True, then the conversion of raw units to mV will be done. Resolution need to be
        specified.

    resolution : int
        Selected resolution for data acquisition.

    device : str
        Specification of the device category.

    plot_result : boolean
        If True it will be presented a graphical representation of the detected burst in the EMG
        signal.

    Returns
    -------
    out : bursts begin (ndarray), bursts end (ndarray)
        Begin and end of bursts (sample number or time instant in seconds).

    smooth_signal: list
        It is returned the smoothed EMG signal (after the processing steps intended to simplify the
        signal).

    threshold_level: float
        The value of the detection threshold used to locate the begin and end of each muscular
        activation period.
    """

    if volts is True:
        if resolution is not None:
            emg_signal = bsnb.raw_to_phy("EMG", device, emg_signal, resolution, option="mV")
            units = "mV"
        else:
            raise RuntimeError(
                "For converting raw units to mV is mandatory the specification of acquisition "
                "resolution.")
    else:
        units = "Input Units"

    if time_units is True:
        time_units_str = "Time (s)"
        time = np.linspace(0, len(emg_signal) / sample_rate, len(emg_signal))
    else:
        time = np.linspace(0, len(emg_signal) - 1, len(emg_signal))
        time_units_str = "Sample Number"

    # ----------------------------------- Baseline Removal -----------------------------------------
    pre_pro_signal = np.array(emg_signal) - np.average(emg_signal)

    # ------------------------------------ Signal Filtering ----------------------------------------
    low_cutoff = 10  # Hz
    #high_cutoff = 300  # Hz

    # Application of the signal to the filter.
    pre_pro_signal = lfilter(*butter(6,low_cutoff,'highpass',fs=Fs),pre_pro_signal)

    # ------------------------------ Application of TKEO Operator ----------------------------------
    tkeo = []
    for i, signal_sample in enumerate(pre_pro_signal):
        if i in (0, len(pre_pro_signal) - 1):
            tkeo.append(signal_sample)
        else:
            tkeo.append(np.power(signal_sample, 2) - (pre_pro_signal[i + 1] *
                                                         pre_pro_signal[i - 1]))

    # Smoothing level - Size of sliding window used during the moving average process (a function
    # of sampling frequency)
    smoothing_level = int((smooth_level / 100) * sample_rate)

    # --------------------------------- Signal Rectification ---------------------------------------
    rect_signal = np.absolute(tkeo)

    # ------------------------------ First Moving Average Filter -----------------------------------
    rect_signal = moving_average(rect_signal, sample_rate / 10)

    # -------------------------------- Second Smoothing Phase --------------------------------------
    smooth_signal = []
    for i in range(0, len(rect_signal)):
        if smoothing_level < i < len(rect_signal) - smoothing_level:
            smooth_signal.append(np.mean(rect_signal[i - smoothing_level:i + smoothing_level]))
        else:
            smooth_signal.append(0)

    # ----------------------------------- Threshold -----------------------------------------------
    avg_pre_pro_signal = np.average(pre_pro_signal)
    std_pre_pro_signal = np.std(pre_pro_signal)

    threshold_level = avg_pre_pro_signal + thres_norm_reg(threshold_level, smooth_signal,
                                                           pre_pro_signal) * std_pre_pro_signal

    # Generation of a square wave reflecting the activation and inactivation periods.
    binary_signal = []
    for i in range(0, len(time)):
        if smooth_signal[i] >= threshold_level:
            binary_signal.append(1)
        else:
            binary_signal.append(0)

    # ------------------------------ Begin and End of Bursts --------------------------------------
    diff_signal = np.diff(binary_signal)
    act_begin = np.where(diff_signal == 1)[0]
    act_end = np.where(diff_signal == -1)[0]

    if time_units is True:
        time_begin = np.array(time)[act_begin]
        time_end = np.array(time)[act_end]
    else:
        time_begin = act_begin
        time_end = act_end

    # If plot is invoked by plot_result flag, then a graphical representation of the R peaks is
    # presented to the user.
    if plot_result is True:
        bsnb.plot([list(time), list(time)], [list(emg_signal), list(np.array(binary_signal) *
                                                               np.max(emg_signal))],
             yAxisLabel=["Data Samples (" + units + ")"] * 2,
             x_axis_label=time_units_str, legend_label=["EMG Signal", "Activation Signal"])

    return time_begin, time_end, smooth_signal, threshold_level

In [38]:
diff_signal = np.diff(binary_signal)
act_begin = np.where(diff_signal == 1)[0]
act_end = np.where(diff_signal == -1)[0]


activation_data = modified_detect_emg_activations(emg_signal, Fs, smooth_level=20, threshold_level=10, time_units=True, volts=False, resolution=None, device="bitalino_rev", plot_result=True)